In [7]:
import time
import requests
from urllib.parse import quote

BASE = "http://leadera1.local"
UNIT = "leaderA1"
EXP  = "sample_and_aliquoting"

def run_stirring(rpm=500):
    url = f"{BASE}/api/workers/{UNIT}/jobs/run/job_name/stirring/experiments/{quote(EXP)}"
    resp = requests.patch(url, json={"options": {"target_rpm": rpm}},
                          headers={"Content-Type": "application/json"})
    print("START:", resp.status_code, resp.text)
    return resp

def update_stirring(rpm):
    url = f"{BASE}/api/workers/{UNIT}/jobs/update/job_name/stirring/experiments/{quote(EXP)}"
    # NOTE: update uses "settings", not "options"
    resp = requests.patch(url, json={"settings": {"target_rpm": rpm}},
                          headers={"Content-Type": "application/json"})
    print("UPDATE:", resp.status_code, resp.text)
    return resp

def list_running():
    url = f"{BASE}/api/workers/{UNIT}/jobs/running"
    r = requests.get(url)
    try:
        txt = r.text
    except Exception:
        txt = ""
    print("RUNNING:", r.status_code, txt)
    return r

def wait_until_stopped(job_name="stirring", timeout_s=15, poll_s=0.5):
    """Poll /jobs/running until job_name disappears or timeout."""
    deadline = time.time() + timeout_s
    url = f"{BASE}/api/workers/{UNIT}/jobs/running"
    while time.time() < deadline:
        r = requests.get(url)
        if r.ok:
            if job_name not in r.text:
                return True
        time.sleep(poll_s)
    return False

def stop_stirring():
    # Preferred “stop this specific job” endpoint
    stop_specific = f"{BASE}/api/workers/{UNIT}/jobs/stop/job_name/stirring/experiments/{quote(EXP)}"
    resp = requests.patch(stop_specific, headers={"Content-Type": "application/json"})
    print("STOP specific:", resp.status_code, getattr(resp, "text", ""))

    if wait_until_stopped():
        print("Confirmed: stirring stopped.")
        return True

    # Fallback #1: stop ALL jobs for this unit in this experiment
    stop_all = f"{BASE}/api/workers/{UNIT}/jobs/stop/experiments/{quote(EXP)}"
    resp2 = requests.patch(stop_all, headers={"Content-Type": "application/json"})
    print("STOP all-in-exp:", resp2.status_code, getattr(resp2, "text", ""))

    if wait_until_stopped():
        print("Confirmed after stop-all: stirring stopped.")
        return True

    # Fallback #2: unit_api stop with query params (works across versions)
    # (Some releases deprecate path-style unit_api stops in favour of query params.)
    unit_api_stop = f"{BASE}/unit_api/jobs/stop?job_name=stirring&experiment={quote(EXP)}"
    resp3 = requests.patch(unit_api_stop, headers={"Content-Type": "application/json"})
    print("STOP unit_api:", resp3.status_code, getattr(resp3, "text", ""))

    ok = wait_until_stopped()
    print("Final stop status:", "stopped" if ok else "still running")
    return ok

if __name__ == "__main__":
    # 1) start
    run_stirring(600)

    # 2) verify running
    list_running()

    # 3) update RPM correctly (uses "settings")
    update_stirring(400)
    time.sleep(10)
    # 4) stop sequence with verification + fallbacks
    stopped = stop_stirring()
    if not stopped:
        # Optional: drop RPM to 0 first, then re-try stop
        update_stirring(0)
        print("Retrying stop after setting RPM=0 …")
        stop_stirring()


START: 202 {"unit":"leaderA1","task_id":"aff832d4-2230-45f1-964d-bcb939d14503","result_url_path":"/unit_api/task_results/aff832d4-2230-45f1-964d-bcb939d14503"}
RUNNING: 202 {"unit":"leaderA1","task_id":"4550beaa-faeb-4015-b123-02dbc4bbeccc","result_url_path":"/unit_api/task_results/4550beaa-faeb-4015-b123-02dbc4bbeccc"}
UPDATE: 202 {"status":"success"}
STOP specific: 202 {"status":"success"}
Confirmed: stirring stopped.


In [ ]:

from chatGpt
http://leadera1.local/api/workers/leadera1/jobs/run/job_name/od_reading/experiments/new_experiment


this one from developers tools
http://leadera1.local/api/workers/leaderA1/jobs/run/job_name/od_reading/experiments/new_experiment

curl -X PATCH \
     -H "Content-Type: application/json" \
     --data '{}' \
http://leadera1.local/api/workers/leaderA1/jobs/stop/job_name/od_reading/experiments/new_experiment


This request resulted in a 415 error code: 
# Start the OD reading job
    start_od_reading_url = f"{base_url}/api/workers/{pioreactor_unit}/jobs/run/job_name/od_reading/experiments/{experiment_name}"
    response = requests.patch(start_od_reading_url)

This request result in a 202 error code: 
    start_od_reading_url = f"{base_url}/api/workers/{pioreactor_unit}/jobs/run/job_name/od_reading/experiments/{experiment_name}"
    response = requests.patch(start_od_reading_url, json={})

Getting sqlite db from leaderA1
    scp pioreactor@leaderA1.local:/home/pioreactor/.pioreactor/storage/pioreactor.sqlite .

To set temperature I had to have help from chatGpt after posting header, response from Network in developers tools

This sets the temperature:
curl -X PATCH \
     -H "Content-Type: application/json" \
     --data '{
       "args": [],
       "env": {
         "EXPERIMENT": "another_temp_test",
         "JOB_SOURCE": "user"
       },
       "options": {
         "automation_name": "thermostat",
         "skip_first_run": 0,
         "target_temperature": 30
       }
     }' \
     http://leadera1.local/api/workers/workerC1/jobs/run/job_name/temperature_automation/experiments/another_temp_test


The script below successfully starts all jobs and can be used for testing. 
#!/Users/harleyking/anaconda3/bin/python3

import requests
import sys


def main():
    # Prompt user for experiment details
    experiment_name = input("Enter experiment name: ")
    experiment_description = input("Enter experiment description (include plasmid name): ")
    organism_strain = input("Enter organism strain: ")
    media_type = input("Enter media type: ")

    # Read pioreactor names from file
    try:
        with open("pioreactor-names.txt", "r") as f:
            pioreactor_names = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        print("Could not find 'pioreactor-names.txt' in the current directory.")
        sys.exit(1)

    base_url = "http://leadera1.local"
    target_rpm = {
        "leaderA1": 1200,
        "workerB1": 200,
        "workerC1": 175,
        "workerD1": 200,
    }

    # Create experiment
    create_experiment_url = f"{base_url}/api/experiments"
    create_experiment_data = {
        "experiment": experiment_name,
        "description": experiment_description,
        "organism_strain": organism_strain,
        "media": media_type,
    }
    response = requests.post(create_experiment_url, json=create_experiment_data)

    if response.status_code == 201:
        print(f"Experiment '{experiment_name}' created successfully.")
    else:
        print(f"Failed to create experiment '{experiment_name}'. Status code: {response.status_code}")
        print(response.text)
        sys.exit(1)

    for pioreactor_unit in pioreactor_names:
        print("\n--- Processing Pioreactor unit:", pioreactor_unit, "---")

        # Assign Pioreactor to experiment
        assign_worker_url = f"{base_url}/api/experiments/{experiment_name}/workers"
        assign_worker_data = {"pioreactor_unit": pioreactor_unit}
        response = requests.put(assign_worker_url, json=assign_worker_data)

        if response.status_code == 200:
            print(f"Pioreactor unit '{pioreactor_unit}' assigned successfully.")
        else:
            print(f"Failed to assign Pioreactor unit '{pioreactor_unit}'. Status code: {response.status_code}")
            print(response.text)
            continue

        # Set temperature to 37C
        set_temperature_url = f"{base_url}/api/workers/{pioreactor_unit}/jobs/run/job_name/temperature_automation/experiments/{experiment_name}"
        temperature_data = {
            "args": [],
            "env": {
                "EXPERIMENT": experiment_name,
                "JOB_SOURCE": "user"
            },
            "options": {
                "automation_name": "thermostat",
                "skip_first_run": 0,
                "target_temperature": 37.0
            }
        }
        response = requests.patch(set_temperature_url, json=temperature_data)

        if response.status_code == 202:
            print(f"Temperature set to 37C for '{pioreactor_unit}'.")
        else:
            print(f"Failed to set temperature on '{pioreactor_unit}'. Status code: {response.status_code}")
            print(response.text)
        
        # Start stirring with specified target RPM
        rpm_value = target_rpm.get(pioreactor_unit, 200)
        start_stirring_url = f"{base_url}/api/workers/{pioreactor_unit}/jobs/run/job_name/stirring/experiments/{experiment_name}"
        stirring_data = {"options": {"target_rpm": rpm_value}}
        response = requests.patch(start_stirring_url, json=stirring_data)

        if response.status_code == 202:
            print(f"Stirring job started successfully on '{pioreactor_unit}' with target RPM of {rpm_value}.")
        else:
            print(f"Failed to start stirring job on '{pioreactor_unit}'. Status code: {response.status_code}")
            print(response.text)

        # Start OD readings
        start_od_reading_url = f"{base_url}/api/workers/{pioreactor_unit}/jobs/run/job_name/od_reading/experiments/{experiment_name}"
        response = requests.patch(start_od_reading_url, json={})

        if response.status_code == 202:
            print(f"OD reading started for '{pioreactor_unit}'.")
        else:
            print(f"Failed to start OD reading on '{pioreactor_unit}'. Status code: {response.status_code}")
            print(response.text)

        # Start growth rate calculation
        growth_rate_url = f"{base_url}/api/workers/{pioreactor_unit}/jobs/run/job_name/growth_rate_calculating/experiments/{experiment_name}"
        growth_rate_data = {
            "args": [],
            "env": {
                "EXPERIMENT": experiment_name,
                "JOB_SOURCE": "user"
            },
            "options": {}
        }
        response = requests.patch(growth_rate_url, json=growth_rate_data)

        if response.status_code == 202:
            print(f"Growth rate calculation started for '{pioreactor_unit}'.")
        else:
            print(f"Failed to start growth rate calculation on '{pioreactor_unit}'. Status code: {response.status_code}")
            print(response.text)

        response = requests.patch(set_temperature_url, json=temperature_data)

        if response.status_code == 202:
            print(f"Temperature set to 37C for '{pioreactor_unit}'.")
        else:
            print(f"Failed to set temperature on '{pioreactor_unit}'. Status code: {response.status_code}")
            print(response.text)

         
    

    print("\nAll Pioreactors are now running the experiment.")


if __name__ == "__main__":
    main()



# http://leadera1.local/api/workers/leaderA1/jobs/run/job_name/growth_rate_calculating/experiments/testing%20api%20script

# http://leadera1.local/api/workers/leaderA1/jobs/run/job_name/temperature_automation/experiments/testing%20api%20script
# http://leadera1.local/api/workers/leaderA1/jobs/run/job_name/temperature_automation/experiments/another_temp_test
# http://leadera1.local/api/workers/workerC1/jobs/run/job_name/temperature_automation/experiments/another_temp_test


# http://leadera1.local/api/workers/$broadcast/jobs/run/job_name/temperature_automation/experiments/temp_automation_test
# http://leadera1.local/api/workers/$broadcast/jobs/update/job_name/temperature_automation/experiments/temp_automation_test

# http://leadera1.local/api/workers/jobs/stop/experiments/testing%20api%20script

This script stops all jobs successfully. 
#!/Users/harleyking/anaconda3/bin/python3

import requests
import json

def get_experiments():
    url = "http://leadera1.local/api/experiments"
    response = requests.get(url)
    return response.json() if response.status_code == 200 else []

def get_assignments():
    url = "http://leadera1.local/api/workers/assignments"
    response = requests.get(url)
    return response.json() if response.status_code == 200 else []

def get_jobs():
    url = "http://leadera1.local/api/contrib/jobs"
    response = requests.get(url)
    return response.json() if response.status_code == 200 else []

def cancel_all_jobs(experiment_name):
    url = f"http://leadera1.local/api/workers/jobs/stop/experiments/{experiment_name}"
    response = requests.patch(url)
    return response.status_code == 202

def main():
    experiments = get_experiments()
    assignments = get_assignments()
    jobs = get_jobs()
    
    if not experiments:
        print("No experiments found.")
        return
    
    # Get the most recent experiment
    current_experiment = experiments[0]['experiment']
    print(f"Currently running experiment: {current_experiment}\n")
    
    # Get assigned Pioreactors
    assigned_pioreactors = [a['pioreactor_unit'] for a in assignments if a['experiment'] == current_experiment]
    print("Assigned Pioreactors:", assigned_pioreactors)
    
    # List jobs running on those Pioreactors
    running_jobs = {pioreactor: [] for pioreactor in assigned_pioreactors}
    for job in jobs:
        job_name = job["job_name"]
        for pioreactor in assigned_pioreactors:
            running_jobs[pioreactor].append(job_name)
    
    print("\nCurrently running jobs:")
    for pioreactor, job_list in running_jobs.items():
        print(f"{pioreactor}: {', '.join(job_list)}")
    
    # Ask user if they want to cancel all jobs
    confirm = input("\nWould you like to cancel all running jobs for this experiment? (yes/no): ").strip().lower()
    if confirm in ['yes']:
        success = cancel_all_jobs(current_experiment)
        status = "Success" if success else "Failed"
        print(f"{status}: Cancelled all jobs for experiment {current_experiment}")
    else:
        print("No jobs were cancelled.")

if __name__ == "__main__":
    main()


#!/Users/harleyking/anaconda3/bin/python3

import requests
import json
from prettytable import PrettyTable

def get_experiments():
    url = "http://leadera1.local/api/experiments"
    response = requests.get(url)
    return response.json() if response.status_code == 200 else []

def get_assignments():
    url = "http://leadera1.local/api/workers/assignments"
    response = requests.get(url)
    return response.json() if response.status_code == 200 else []

def cancel_all_jobs(experiment_name):
    url = f"http://leadera1.local/api/workers/jobs/stop/experiments/{experiment_name}"
    response = requests.patch(url)
    return response.status_code == 202

def main():
    experiments = get_experiments()
    assignments = get_assignments()
    
    if not experiments:
        print("No experiments found.")
        return
    
    # Filter experiments with assigned pioreactors
    experiment_pioreactor_map = {}
    for assignment in assignments:
        experiment = assignment['experiment']
        pioreactor = assignment['pioreactor_unit']
        position = pioreactor.replace("leaderA", "A").replace("worker", "").upper()
        if experiment in experiment_pioreactor_map:
            experiment_pioreactor_map[experiment].append(position)
        else:
            experiment_pioreactor_map[experiment] = [position]
    
    # Generate table
    table = PrettyTable()
    table.field_names = ["Experiment Number", "Experiment Name", "Assigned Pioreactors"]
    
    running_experiments = list(experiment_pioreactor_map.keys())
    for i, experiment in enumerate(running_experiments, 1):
        table.add_row([i, experiment, ", ".join(experiment_pioreactor_map[experiment])])
    
    print(table)
    
    # Get user input
    choice = input("Enter a number e.g. 1, 2 or 'all' to stop pioreactor jobs: ").strip().lower()
    if choice == "all":
        for experiment in running_experiments:
            success = cancel_all_jobs(experiment)
            status = "Success" if success else "Failed"
            print(f"{status}: Cancelled all jobs for experiment {experiment}")
    elif choice.isdigit() and 1 <= int(choice) <= len(running_experiments):
        selected_experiment = running_experiments[int(choice) - 1]
        success = cancel_all_jobs(selected_experiment)
        status = "Success" if success else "Failed"
        print(f"{status}: Cancelled all jobs for experiment {selected_experiment}")
    else:
        print("Invalid input. Please enter a valid experiment number or 'all'.")

if __name__ == "__main__":
    main()
